# مسئلهٔ ۱ — V2 / مرحلهٔ ۲ و ۳: Freeze کردن split و ساخت Sequence Manifest

ورودی این نوت‌بوک، `video_manifest_v2.csv` مرحلهٔ audit است. خروجی، دقیقاً یک sequence پنج‌ثانیه‌ای و ۱۶ timestamp قطعی برای هر ویدئوی سالم است.

- مثبت: پنجرهٔ V2-W2 برابر `[time_of_event - 3s, time_of_event + 2s]`، با جابه‌جایی محافظه‌کارانه در مرز ویدئو تا طول آن دقیقاً ۵ ثانیه بماند.
- منفی: پنجرهٔ ۵ثانیه‌ای که موقعیت نسبی آن با یک ویدئوی مثبت از همان split تطبیق داده شده است.
- timestampها در واحد ثانیه از ابتدای MP4 هستند؛ هنوز هیچ فریمی استخراج یا augmentation اعمال نمی‌شود.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd

DATA_ROOT = Path(r'P:\NexarCollisionData')
VIDEO_MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
FROZEN_SPLIT_PATH = DATA_ROOT / 'metadata_split_v1.csv'
SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
EXCLUDED_PATH = DATA_ROOT / 'sequence_manifest_v2_excluded.csv'
SUMMARY_PATH = DATA_ROOT / 'sequence_manifest_v2_summary.json'

WINDOW_VERSION = 'V2-W2'
WINDOW_LENGTH_SECONDS = 5.0
POSITIVE_PRE_EVENT_SECONDS = 3.0
POSITIVE_POST_EVENT_SECONDS = 2.0
NUM_FRAMES = 16
SPLIT_SEED = 42
MATCHING_SEED = 42

assert POSITIVE_PRE_EVENT_SECONDS + POSITIVE_POST_EVENT_SECONDS == WINDOW_LENGTH_SECONDS
assert VIDEO_MANIFEST_PATH.exists(), f'Run notebook 07 first: {VIDEO_MANIFEST_PATH}'
print(f'Data root: {DATA_ROOT}')

Data root: P:\NexarCollisionData


In [2]:
required_columns = {
    'video_id', 'video_path', 'label', 'time_of_event', 'duration',
    'weather', 'light_conditions', 'scene', 'split', 'is_valid', 'error_reason',
}
manifest = pd.read_csv(VIDEO_MANIFEST_PATH)
assert required_columns.issubset(manifest.columns), sorted(required_columns - set(manifest.columns))

manifest = manifest.copy()
manifest['video_id'] = manifest['video_id'].astype(str)
manifest['label'] = manifest['label'].astype(int)
manifest['duration'] = pd.to_numeric(manifest['duration'], errors='coerce')
manifest['time_of_event'] = pd.to_numeric(manifest['time_of_event'], errors='coerce')
manifest['is_valid'] = manifest['is_valid'].astype(str).str.lower().eq('true')

assert len(manifest) == 600
assert manifest['video_id'].is_unique
assert manifest['split'].isin(['train', 'validation']).all()
assert (manifest['label'].eq(1) == manifest['time_of_event'].notna()).all(), 'Label must come only from time_of_event'

# This is a frozen copy, not a new random split. Future V2 notebooks read this file.
frozen_split = manifest[['video_id', 'video_path', 'label', 'split']].copy()
frozen_split['split_seed'] = SPLIT_SEED
frozen_split['split_source'] = 'inherited_from_video_splits.csv'
frozen_split['audit_manifest'] = VIDEO_MANIFEST_PATH.name
frozen_split = frozen_split.sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)
frozen_split.to_csv(FROZEN_SPLIT_PATH, index=False)

print(f'Frozen split saved: {FROZEN_SPLIT_PATH}')
display(pd.crosstab(frozen_split['split'], frozen_split['label']))

Frozen split saved: P:\NexarCollisionData\metadata_split_v1.csv


label,0,1
split,,
train,240,240
validation,60,60


In [3]:
def fixed_length_window(requested_start: float, duration: float) -> tuple[float, float, str]:
    """Shift, rather than shorten, a 5-second window at video boundaries."""
    if not np.isfinite(duration) or duration < WINDOW_LENGTH_SECONDS:
        raise ValueError('video_shorter_than_required_window')

    max_start = duration - WINDOW_LENGTH_SECONDS
    start = float(np.clip(requested_start, 0.0, max_start))
    if np.isclose(start, requested_start):
        policy = 'requested_window_kept'
    elif requested_start < 0:
        policy = 'shifted_to_video_start'
    else:
        policy = 'shifted_to_video_end'
    return start, start + WINDOW_LENGTH_SECONDS, policy

def timestamps_for_window(start: float, end: float) -> np.ndarray:
    return np.linspace(start, end, num=NUM_FRAMES, endpoint=False, dtype=np.float64)

def build_positive_sequence(row: pd.Series) -> dict:
    event_time = float(row.time_of_event)
    duration = float(row.duration)
    requested_start = event_time - POSITIVE_PRE_EVENT_SECONDS
    start, end, boundary_policy = fixed_length_window(requested_start, duration)
    if not (start <= event_time <= end):
        raise ValueError('event_not_contained_after_boundary_shift')

    record = {
        'sequence_id': f'{WINDOW_VERSION}_{row.video_id}',
        'video_id': row.video_id,
        'video_path': row.video_path,
        'label': int(row.label),
        'split': row.split,
        'time_of_event': event_time,
        'duration': duration,
        'window_start': start,
        'window_end': end,
        'window_length_seconds': end - start,
        'window_center_ratio': (start + WINDOW_LENGTH_SECONDS / 2) / duration,
        'window_policy': f'positive_event_centered;{boundary_policy}',
        'matched_positive_video_id': row.video_id,
        'matching_seed': MATCHING_SEED,
        'sampling_seed': MATCHING_SEED,
        'num_frames': NUM_FRAMES,
        'preprocessing_version': f'{WINDOW_VERSION}_sequence_manifest',
        'weather': row.weather,
        'light_conditions': row.light_conditions,
        'scene': row.scene,
    }
    for index, timestamp in enumerate(timestamps_for_window(start, end)):
        record[f'timestamp_{index:02d}'] = float(timestamp)
    return record


In [4]:
# Keep exclusions explicit. With the audited 600 videos this table should be empty.
eligible = manifest.loc[manifest['is_valid'] & manifest['duration'].ge(WINDOW_LENGTH_SECONDS)].copy()
excluded = manifest.loc[~manifest.index.isin(eligible.index)].copy()
excluded['sequence_exclusion_reason'] = np.where(
    ~excluded['is_valid'],
    excluded['error_reason'].fillna('invalid_video'),
    'video_shorter_than_required_window',
)
excluded.to_csv(EXCLUDED_PATH, index=False)

assert len(eligible) == 600, f'Unexpected exclusions; inspect {EXCLUDED_PATH}'
assert eligible.groupby(['split', 'label']).size().eq(
    pd.Series({('train', 0): 240, ('train', 1): 240, ('validation', 0): 60, ('validation', 1): 60})
).all()

positive_rows = eligible.loc[eligible['label'].eq(1)].copy()
positive_sequences = [build_positive_sequence(row) for _, row in positive_rows.iterrows()]
positive_sequence_df = pd.DataFrame(positive_sequences)
assert len(positive_sequence_df) == 300
display(positive_sequence_df[['split', 'video_id', 'time_of_event', 'duration', 'window_start', 'window_end', 'window_policy']].head())

,split,video_id,time_of_event,duration,window_start,window_end,window_policy
0,train,0,20.760,40.069204,17.760,22.760,positive_event_centered;requested_window_kept
1,train,4,19.367,40.066667,16.367,21.367,positive_event_centered;requested_window_kept
2,train,5,20.874,40.098039,17.874,22.874,positive_event_centered;requested_window_kept
3,train,6,19.233,39.966667,16.233,21.233,positive_event_centered;requested_window_kept
4,train,10,20.900,40.466667,17.900,22.900,positive_event_centered;requested_window_kept


In [5]:
def build_negative_sequences(negative_rows: pd.DataFrame, positives: pd.DataFrame, split_name: str) -> list[dict]:
    """Pair each negative with a positive relative window position from the same frozen split."""
    negative_rows = negative_rows.sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)
    positives = positives.loc[positives['split'].eq(split_name)].sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)
    assert len(negative_rows) == len(positives), f'Unbalanced {split_name} split'

    split_offset = 0 if split_name == 'train' else 10_000
    rng = np.random.default_rng(MATCHING_SEED + split_offset)
    paired_positive_indices = rng.permutation(len(positives))
    records = []

    for negative_idx, (_, row) in enumerate(negative_rows.iterrows()):
        matched_positive = positives.iloc[paired_positive_indices[negative_idx]]
        duration = float(row.duration)
        target_center_ratio = float(matched_positive.window_center_ratio)
        requested_start = target_center_ratio * duration - WINDOW_LENGTH_SECONDS / 2
        start, end, boundary_policy = fixed_length_window(requested_start, duration)

        record = {
            'sequence_id': f'{WINDOW_VERSION}_{row.video_id}',
            'video_id': row.video_id,
            'video_path': row.video_path,
            'label': int(row.label),
            'split': row.split,
            'time_of_event': np.nan,
            'duration': duration,
            'window_start': start,
            'window_end': end,
            'window_length_seconds': end - start,
            'window_center_ratio': (start + WINDOW_LENGTH_SECONDS / 2) / duration,
            'window_policy': f'negative_matched_relative_position;{boundary_policy}',
            'matched_positive_video_id': matched_positive.video_id,
            'matching_seed': MATCHING_SEED + split_offset,
            'sampling_seed': MATCHING_SEED,
            'num_frames': NUM_FRAMES,
            'preprocessing_version': f'{WINDOW_VERSION}_sequence_manifest',
            'weather': row.weather,
            'light_conditions': row.light_conditions,
            'scene': row.scene,
        }
        for index, timestamp in enumerate(timestamps_for_window(start, end)):
            record[f'timestamp_{index:02d}'] = float(timestamp)
        records.append(record)
    return records

negative_sequences = []
for split_name in ('train', 'validation'):
    split_negatives = eligible.loc[eligible['label'].eq(0) & eligible['split'].eq(split_name)].copy()
    negative_sequences.extend(build_negative_sequences(split_negatives, positive_sequence_df, split_name))

negative_sequence_df = pd.DataFrame(negative_sequences)
assert len(negative_sequence_df) == 300
display(negative_sequence_df[['split', 'video_id', 'matched_positive_video_id', 'duration', 'window_start', 'window_end', 'window_policy']].head())

,split,video_id,matched_positive_video_id,duration,window_start,window_end,window_policy
0,train,1046,1000,40.295082,15.860662,20.860662,negative_matched_relative_position;requested_w...
1,train,1049,455,40.228758,16.421544,21.421544,negative_matched_relative_position;requested_w...
2,train,1054,580,39.112628,17.381170,22.381170,negative_matched_relative_position;requested_w...
3,train,1060,309,40.066890,16.201986,21.201986,negative_matched_relative_position;requested_w...
4,train,1072,450,41.033333,17.730001,22.730001,negative_matched_relative_position;requested_w...


In [6]:
sequence_manifest = pd.concat([positive_sequence_df, negative_sequence_df], ignore_index=True)
sequence_manifest = sequence_manifest.sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)
timestamp_columns = [f'timestamp_{index:02d}' for index in range(NUM_FRAMES)]

assert len(sequence_manifest) == 600
assert sequence_manifest['sequence_id'].is_unique
assert sequence_manifest['video_id'].is_unique
assert sequence_manifest.groupby('video_id')['split'].nunique().eq(1).all()
assert np.allclose(sequence_manifest['window_length_seconds'], WINDOW_LENGTH_SECONDS)
assert (sequence_manifest['window_start'] >= -1e-8).all()
assert (sequence_manifest['window_end'] <= sequence_manifest['duration'] + 1e-8).all()
assert (sequence_manifest[timestamp_columns].diff(axis=1).iloc[:, 1:] > 0).all().all()

positive_sequences_check = sequence_manifest.loc[sequence_manifest['label'].eq(1)]
assert ((positive_sequences_check['window_start'] <= positive_sequences_check['time_of_event']) &
        (positive_sequences_check['time_of_event'] <= positive_sequences_check['window_end'])).all()

sequence_manifest.to_csv(SEQUENCE_MANIFEST_PATH, index=False)

summary = {
    'window_version': WINDOW_VERSION,
    'window_length_seconds': WINDOW_LENGTH_SECONDS,
    'positive_window_definition': 'time_of_event - 3s to time_of_event + 2s; shifted at boundaries',
    'negative_window_definition': '5-second window matched to a positive relative window-center position within the same split',
    'num_sequences': int(len(sequence_manifest)),
    'num_frames_per_sequence': NUM_FRAMES,
    'timestamp_unit': 'seconds from video start',
    'matching_seed': MATCHING_SEED,
    'excluded_videos': int(len(excluded)),
    'split_class_counts': {
        f'{split}|{label}': int(count)
        for (split, label), count in sequence_manifest.groupby(['split', 'label']).size().items()
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Sequence manifest saved: {SEQUENCE_MANIFEST_PATH}')
print(f'Frozen split saved: {FROZEN_SPLIT_PATH}')
print(f'Excluded-video report: {EXCLUDED_PATH}')
print(f'Summary: {SUMMARY_PATH}')
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))
display(sequence_manifest[['sequence_id', 'video_id', 'label', 'split', 'window_start', 'window_end', 'matched_positive_video_id', *timestamp_columns[:4]]].head(10))

Sequence manifest saved: P:\NexarCollisionData\sequence_manifest_v2.csv
Frozen split saved: P:\NexarCollisionData\metadata_split_v1.csv
Excluded-video report: P:\NexarCollisionData\sequence_manifest_v2_excluded.csv
Summary: P:\NexarCollisionData\sequence_manifest_v2_summary.json


label,0,1
split,,
train,240,240
validation,60,60


,sequence_id,video_id,label,split,window_start,window_end,matched_positive_video_id,timestamp_00,timestamp_01,timestamp_02,timestamp_03
0,V2-W2_0,0,1,train,17.760,22.760,0,17.760,18.0725,18.385,18.6975
1,V2-W2_4,4,1,train,16.367,21.367,4,16.367,16.6795,16.992,17.3045
2,V2-W2_5,5,1,train,17.874,22.874,5,17.874,18.1865,18.499,18.8115
3,V2-W2_6,6,1,train,16.233,21.233,6,16.233,16.5455,16.858,17.1705
4,V2-W2_10,10,1,train,17.900,22.900,10,17.900,18.2125,18.525,18.8375
5,V2-W2_14,14,1,validation,16.067,21.067,14,16.067,16.3795,16.692,17.0045
6,V2-W2_15,15,1,train,16.700,21.700,15,16.700,17.0125,17.325,17.6375
7,V2-W2_25,25,1,train,17.033,22.033,25,17.033,17.3455,17.658,17.9705
8,V2-W2_29,29,1,validation,16.933,21.933,29,16.933,17.2455,17.558,17.8705
9,V2-W2_31,31,1,validation,16.100,21.100,31,16.100,16.4125,16.725,17.0375


## شرط پایان این مرحله

باید ۶۰۰ sequence، بدون exclusion، با split ثابت ۴۸۰/۱۲۰ و ۱۶ timestamp صعودی در هر ردیف داشته باشیم. مرحلهٔ بعدی فقط frameهای همین timestampها را decode و به‌شکل cache قطعی ذخیره می‌کند.

در inference نهایی برای یک MP4 جدید، از `time_of_event` استفاده نمی‌شود: پنجره‌های پنج‌ثانیه‌ای لغزان از کل ویدئو ساخته می‌شوند، هر پنجره امتیاز می‌گیرد و سپس احتمال‌ها به یک تصمیم video-level تجمیع می‌شوند.